# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_dict = json.loads(dataset.metadata.to_json_ld())

print('Title:', dataset.metadata.name)
print('Description:', dataset.metadata.description)

## 2. Data Overview
Review available record sets, their fields, columns, and their `@id` values.

All entities are referenced by their `@id` fields.

In [ ]:
# List available record sets, their ids, and associated fields
def get_all_record_sets(md):
    if 'recordSet' in md:
        # If no record sets
        if not md['recordSet']:
            print('No record sets defined in metadata.')
            return []
        rs = md['recordSet']
        # If this is a single record set, wrap it
        if isinstance(rs, dict):
            rs = [rs]
        return rs
    print('No `recordSet` key found in metadata.')
    return []

# Attempt to parse record sets and display their fields
record_sets = []
# The Croissant metadata root is usually a dict with '@graph' key holding all objects
if '@graph' in metadata_dict:
    # Find record sets by type
    for item in metadata_dict['@graph']:
        if item.get('@type') == 'cr:RecordSet' or item.get('@type') == 'RecordSet':
            record_sets.append(item)
else:
    # Fallback if @graph missing
    record_sets = get_all_record_sets(metadata_dict)

if len(record_sets) == 0:
    print('No record sets found. The dataset may define only one data file as the default record set.')
    # Try to retrieve first distribution object as a record set
    # Print the available top-level metadata keys
    if 'distribution' in metadata_dict:
        print(f"Found 'distribution' (likely data file(s)) in metadata: {metadata_dict['distribution']}")
    else:
        print('No "distribution" in metadata. The dataset may be metadata only.')
else:
    for rs in record_sets:
        print(f"RecordSet: @id={rs.get('@id')}, name={rs.get('name')}")
        # List its fields/columns if present
        if 'field' in rs:
            fields = rs['field']
            if isinstance(fields, dict):
                fields = [fields]
            for f in fields:
                print(f"  Field: @id={f.get('@id')}, name={f.get('name')}")
        if 'column' in rs:
            columns = rs['column']
            if isinstance(columns, dict):
                columns = [columns]
            for c in columns:
                print(f"  Column: @id={c.get('@id')}, name={c.get('name')}")

## 3. Data Extraction
Load data from a specific record set (by its `@id`) into a DataFrame for analysis.

We use the record set and field `@id` values discovered above (manually inspect and update here as needed).

In [ ]:
# If no record sets are declared, mlcroissant uses the main data table as the default.
# We'll try to get one record set. Update this cell accordingly for your dataset.

# Try to infer the record set @id
if len(record_sets) > 0:
    # Use the first record set
    main_record_set_id = record_sets[0]['@id']
    print(f"Using record set: {main_record_set_id}")
else:
    # If no record set, Croissant standard: use the '@id' of the dataset itself (file-based)
    main_record_set_id = metadata_dict['@id']
    print(f"No explicit record set, using dataset @id: {main_record_set_id}")

# If you know additional record set ids, add here:
record_sets_ids = [main_record_set_id]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Fields in {record_set_id}: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data manipulation/analysis steps using field `@id` (or column name) from above.

- Filtering
- Normalization of numeric field
- Grouping/categorization by key field

All references to columns use their `@id` where relevant, or Pandas column name if `@id` is not used as column header.

In [ ]:
# Edit your numeric and grouping fields based on those printed in section 3

df = dataframes[main_record_set_id]

# Example field names likely present: 'MW (kDa)', 'Sequence Coverage %', 'Unique Peptide', etc.
# Let's try with molecular weight if available

numeric_field = None
possible_numeric_fields = ['MW (kDa)', 'Molecular Weight', 'Sequence Coverage %', 'Unique Peptide', 'coverage', 'Abundance']
for col in df.columns:
    if col in possible_numeric_fields:
        numeric_field = col
        break

if numeric_field is None:
    print('No numeric field found in columns. Please assign a numeric field manually.')
else:
    print(f'Using numeric field for EDA: {numeric_field}')

    # Remove non-numeric values if needed
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Filtering: get records with value > threshold
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f'Filtered records with {numeric_field} > {threshold}:')
    print(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Group by a categorical field
    group_field = None
    possible_group_fields = ['Description', 'Type', 'Protein Group', 'Sample', 'Experiment']
    for col in df.columns:
        if col in possible_group_fields:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print('No suitable group field found for grouping step.')

## 5. Visualization
Visualize the filtered and/or grouped dataset. This helps understand the distribution or key relationships.

In [ ]:
import matplotlib.pyplot as plt

# Show histogram of the numeric field (if available)
if numeric_field is not None and df[numeric_field].notnull().any():
    plt.figure(figsize=(8, 4))
    df[numeric_field].dropna().hist(bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group_field is present and grouped_df exists, show bar plot
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10, 4))
        grouped_df.set_index(group_field)[numeric_field].plot(kind='bar')
        plt.title(f'Mean {numeric_field} by {group_field} (> mean filter)')
        plt.show()
else:
    print('No numeric field for visualization.')

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset via its Croissant schema, reviewed both the metadata and data content using the `mlcroissant` library, and performed basic exploratory data analysis (EDA). The field and grouping operations may be adjusted based on the actual fields listed in your dataset. Further domain-specific analysis and modeling can build off this workflow.

**Summary:**
- Croissant metadata is machine-readable and makes systematic FAIR exploration possible
- Data can be loaded and processed directly by referencing `@id` of record sets and fields
- This approach allows for dynamic data connection and robust data provenance tracking.

Refer to domain documentation and dataset description in the metadata for detailed interpretation of protein fields and scientific context.